# Descarga e importación de líbrerias


In [4]:
%pip install --upgrade pip
%pip install selenium
%pip install pywin32

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:


import pandas as pd #Para la manipulación de datos
from datetime import datetime  #Para la fecha de hoy
from selenium import webdriver #Para la navegación automática
import os #Para la manipulación de archivos
import time #Para la espera
import win32com.client as win32

#Importación de líbrerias de Selenium
from selenium.webdriver.common.by import By #Para la localización de elementos
from selenium.webdriver.support.ui import WebDriverWait #Para la espera de carga de elementos
from selenium.webdriver.support import expected_conditions as EC #Para la espera de carga de elementos
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
import time
import shutil

# Descargar archivo de SNIES



In [15]:
#Obtenemos el directorio actual
current_directory = os.getcwd()

#Función para clickear de forma segura
def safe_click(driver, locator):
    try:
        #Esperar a que cargue el elemento
        element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(locator))
        element.click()
    except (StaleElementReferenceException, TimeoutException):
        # Intentar nuevamente
        try:
            element = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(locator))
            element.click()
        except (StaleElementReferenceException, TimeoutException):
            print(f"Failed to click element {locator}")

#Configuración del driver
options = webdriver.ChromeOptions() # Usamos chrome
prefs = {"download.default_directory" : os.path.join(current_directory, "Programas")}  # Descargar archivos dentro de esta carpeta

options.add_argument('--no-sandbox') # Seguridad
options.add_argument('--user-agent=""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.157 Safari/537.36""') # user agent
options.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=options)

#Navegamos a la página del SNIES
driver.get("https://hecaa.mineducacion.gov.co/consultaspublicas/programas")

driver.implicitly_wait(5) #Espera 5 segundos para cargar la página

##//*[@id="formFiltro:j_idt35"]/tbody/tr[2]/td/div/div[2]/span
#Identificadores de elementos a clickear
institucionFilter = (By.XPATH, '//*[@id="formFiltro:j_idt35"]/tbody/tr[2]/td/div/div[2]/span') #Filtro de estado de institución (Activo)
programaFilter = (By.XPATH, '//*[@id="formFiltro:j_idt68"]/tbody/tr[2]/td/div/div[2]/span') #Filtro de estado de programa (Activo)
academicoFilter = (By.XPATH, '//*[@id="formFiltro:j_idt109"]/tbody/tr[2]/td/div/div[2]/span') #Filtro de tipo académico (Pregrado)
formacionFilter = (By.XPATH, '//*[@id="formFiltro:j_idt116"]/tbody/tr[4]/td/div/div[2]/span') #Filtro de tipo de formación (Universitario)
downloadButton = (By.XPATH, '//*[@id="j_idt169:j_idt171"]') #Botón de descarga

#Clickeamos los filtros
safe_click(driver, institucionFilter)
time.sleep(3)
safe_click(driver, programaFilter)
safe_click(driver, academicoFilter)
safe_click(driver, formacionFilter)
safe_click(driver, downloadButton)

# Definimos la ruta de forma segura para evitar el SyntaxWarning
# Usamos os.path.join para que Python maneje las barras correctamente
ruta_esperada = os.path.join(current_directory, "Programas", "Programas.xlsx")

print(f"Buscando el archivo en: {ruta_esperada}")

# Añadimos un contador para que no sea infinito
segundos_esperados = 0
maximo_espera = 60 # Se rinde después de 1 minuto si no aparece

while not os.path.exists(ruta_esperada) and segundos_esperados < maximo_espera:
    print(f"Descargando archivo... ({segundos_esperados}s)")
    time.sleep(5)
    segundos_esperados += 5

if os.path.exists(ruta_esperada):
    print("¡Éxito! El archivo base ya está en la carpeta.")
    
    # 1. Definimos la fecha de hoy (formato DD-MM-YY para que coincida con tu lectura)
    today_str = datetime.today().strftime('%d-%m-%y')
    ruta_con_fecha = os.path.join(current_directory, "Programas", f"Programas {today_str}.xlsx")
    
    try:
        # 2. Si ya existía un archivo de hoy, lo borramos para evitar conflictos
        if os.path.exists(ruta_con_fecha):
            os.remove(ruta_con_fecha)
            
        # 3. Renombramos: de 'Programas.xlsx' a 'Programas 24-03-26.xlsx'
        shutil.move(ruta_esperada, ruta_con_fecha)
        print(f"✅ Archivo sellado y listo: {os.path.basename(ruta_con_fecha)}")
    except Exception as e:
        print(f"❌ Error al renombrar: {e}")
        
else:
    print("ERROR: El archivo no apareció. Revisa si el navegador realmente inició la descarga.")

driver.quit()



Buscando el archivo en: d:\SNIES INACTIVOS-NUEVOS-MODIFICADOS-20260406T135829Z-1-001\SNIES INACTIVOS-NUEVOS-MODIFICADOS\SNIES PREGRADO\Programas\Programas\Programas.xlsx
Descargando archivo... (0s)
¡Éxito! El archivo base ya está en la carpeta.
✅ Archivo sellado y listo: Programas 07-04-26.xlsx


# Archivo a Dataframe


In [16]:
import glob
import re

# 1. Fecha de hoy
today = datetime.today().strftime('%d-%m-%y')

# 2. BUSCAR DINÁMICAMENTE EL ÚLTIMO ARCHIVO
path_programas = os.path.join(current_directory, "Programas", "Programas *.xlsx")
archivos_existentes = glob.glob(path_programas)

# Extraemos las fechas de los nombres de archivos para encontrar la más reciente
fechas_encontradas = []
for f in archivos_existentes:
    match = re.search(r'(\d{2}-\d{2}-\d{2})', os.path.basename(f))
    if match:
        fecha_str = match.group(1)
        if fecha_str != today: # No queremos compararlo consigo mismo si ya se descargó hoy
            fechas_encontradas.append(datetime.strptime(fecha_str, '%d-%m-%y'))

if fechas_encontradas:
    last = max(fechas_encontradas).strftime('%d-%m-%y')
    print(f"Último archivo detectado automáticamente: {last}")
else:
    last = None
    print("No se encontró un archivo anterior para comparar.")

# 3. Continuar con tu proceso de renombrar el descargado
programaNombre = os.path.join(current_directory, "Programas", "Programas.xlsx")
# ... resto de tu código

#Leemos el archivo
df = pd.read_excel("Programas\Programas "+ today +".xlsx", sheet_name="Programas")

#Leemos el archivo más (Si Existe)
try:
    dfpast = pd.read_excel("Programas\Programas "+ last +".xlsx", sheet_name="Programas")
    #dfpast=pd.read_excel('Programas 23-02-24.xlsx', sheet_name="Programas")
except:
    print("No existe archivo antiguo para comparar")

#Leemos el archivo de categorización de divisiones Uninorte
categories = pd.read_excel("Categorización divisiones SNIES .xlsx", sheet_name="Hoja3")

#Columnas a usar para el análisis
finalColumns = [
    'CÓDIGO_INSTITUCIÓN',
    'NOMBRE_INSTITUCIÓN',
    'SECTOR',
    'DEPARTAMENTO_OFERTA_PROGRAMA',
    'MUNICIPIO_OFERTA_PROGRAMA',
    'CÓDIGO_SNIES_DEL_PROGRAMA',
    'NOMBRE_DEL_PROGRAMA',
    'MODALIDAD',
    'NÚMERO_CRÉDITOS',
    'NÚMERO_PERIODOS_DE_DURACIÓN',
    'PERIODICIDAD',
    'COSTO_MATRÍCULA_ESTUD_NUEVOS',
    'PERIODICIDAD_ADMISIONES',
    'FECHA_DE_REGISTRO_EN_SNIES',
    'CINE_F_2013_AC_CAMPO_AMPLIO', 
    'CINE_F_2013_AC_CAMPO_ESPECÍFIC',
    'CINE_F_2013_AC_CAMPO_DETALLADO',
    'NÚCLEO_BÁSICO_DEL_CONOCIMIENTO']


droppedDf = df[finalColumns] #Filtramos las columnas del archivo más reciente
droppedDfpast = dfpast[finalColumns] #Filtramos las columnas del archivo más antiguo

#Quitamos el aviso del SNIES
droppedDf = droppedDf.iloc[:-2]
droppedDfpast = droppedDfpast.iloc[:-2]


<>:32: SyntaxWarning: invalid escape sequence '\P'
<>:36: SyntaxWarning: invalid escape sequence '\P'
<>:32: SyntaxWarning: invalid escape sequence '\P'
<>:36: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ARACATACA\AppData\Local\Temp\ipykernel_18792\577586478.py:32: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_excel("Programas\Programas "+ today +".xlsx", sheet_name="Programas")
C:\Users\ARACATACA\AppData\Local\Temp\ipykernel_18792\577586478.py:36: SyntaxWarning: invalid escape sequence '\P'
  dfpast = pd.read_excel("Programas\Programas "+ last +".xlsx", sheet_name="Programas")
c:\Users\ARACATACA\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Último archivo detectado automáticamente: 06-04-26


c:\Users\ARACATACA\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\ARACATACA\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


# Depuración de datos


In [17]:
#Eliminamos los programas sin código de programa y convertimos a entero
droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'] = droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'].dropna().astype(int)
droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'] = droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'].dropna().astype(int)

#Rellenamos los valores nulos con 0 y convertimos a entero
droppedDf['NÚMERO_CRÉDITOS'] = droppedDf['NÚMERO_CRÉDITOS'].fillna(0).astype(int)
droppedDfpast['NÚMERO_CRÉDITOS'] = droppedDfpast['NÚMERO_CRÉDITOS'].fillna(0).astype(int)

#Creamos un código para identificar el programa en caso que cambie el número de créditos
droppedDf["PROGINSTI"] = droppedDf["CÓDIGO_SNIES_DEL_PROGRAMA"].astype(str)+"-"+droppedDf["NÚMERO_CRÉDITOS"].astype(str)
droppedDfpast["PROGINSTI"] =droppedDfpast["CÓDIGO_SNIES_DEL_PROGRAMA"].astype(str)+"-"+droppedDfpast["NÚMERO_CRÉDITOS"].astype(str)

#Fecha de obtención
newDate = pd.to_datetime(today, format='%d-%m-%y')
newDateFormatted = newDate.strftime('%d/%m/%Y')


# Creación Nuevos/Modificados/Inactivos

In [18]:
# 1. IDENTIFICACIÓN POR EXISTENCIA (Basado solo en el Código SNIES)
snies_hoy = set(droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'])
snies_ayer = set(droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'])

# Programas que NO estaban ayer y están hoy
nuevos_ids = snies_hoy - snies_ayer
# Programas que ESTABAN ayer y ya no están hoy
inactivos_ids = snies_ayer - snies_hoy
# Programas que están en AMBOS (Candidatos a ser "Modificados")
comunes_ids = snies_hoy & snies_ayer

# Creamos los DataFrames iniciales
nuevosDF = droppedDf[droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'].isin(nuevos_ids)].copy()
eliminadosDF = droppedDfpast[droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'].isin(inactivos_ids)].copy()

# 2. DETECCIÓN DE MODIFICADOS (Comparación de Atributos)
# Filtramos los programas que persisten para comparar sus datos internos
df_comunes_hoy = droppedDf[droppedDf['CÓDIGO_SNIES_DEL_PROGRAMA'].isin(comunes_ids)]
df_comunes_ayer = droppedDfpast[droppedDfpast['CÓDIGO_SNIES_DEL_PROGRAMA'].isin(comunes_ids)]

# Unimos ambos estados en una sola tabla comparativa
comparativa = df_comunes_hoy.merge(
    df_comunes_ayer, 
    on='CÓDIGO_SNIES_DEL_PROGRAMA', 
    suffixes=('_NUEVO', '_ANTIGUO')
)

# Definimos las columnas que queremos vigilar
columnas_vigilar = ['MODALIDAD', 'NÚMERO_CRÉDITOS', 'COSTO_MATRÍCULA_ESTUD_NUEVOS', 'MUNICIPIO_OFERTA_PROGRAMA']

# Filtramos donde al menos uno de los valores haya cambiado
mascara_cambio = False
for col in columnas_vigilar:
    # fillna('') evita que los valores nulos rompan la comparación
    mascara_cambio |= (comparativa[f'{col}_NUEVO'].fillna('') != comparativa[f'{col}_ANTIGUO'].fillna(''))

modificadosDF = comparativa[mascara_cambio].copy()

# 3. FORMATEO FINAL Y FECHAS
for df_temp in [nuevosDF, eliminadosDF, modificadosDF]:
    df_temp['FECHA_OBTENCION'] = newDateFormatted
    if 'FECHA_DE_REGISTRO_EN_SNIES' in df_temp.columns:
        df_temp["FECHA_DE_REGISTRO_EN_SNIES"] = pd.to_datetime(df_temp["FECHA_DE_REGISTRO_EN_SNIES"]).dt.date

# Limpieza de nombres de columnas para modificados (opcional pero recomendado)
# Esto ayuda a que el Excel de "Modificados" se vea más limpio

In [19]:
#Chequeo de longitudes
print(len(droppedDf))
print(len(droppedDfpast))
print(len(eliminadosDF))
print(len(nuevosDF))
print(len(modificadosDF))


5246
5244
0
2
4


In [14]:
print(modificadosDF.columns.tolist())

['CÓDIGO_INSTITUCIÓN_NUEVO', 'NOMBRE_INSTITUCIÓN_NUEVO', 'SECTOR_NUEVO', 'DEPARTAMENTO_OFERTA_PROGRAMA_NUEVO', 'MUNICIPIO_OFERTA_PROGRAMA_NUEVO', 'CÓDIGO_SNIES_DEL_PROGRAMA', 'NOMBRE_DEL_PROGRAMA_NUEVO', 'MODALIDAD_NUEVO', 'NÚMERO_CRÉDITOS_NUEVO', 'NÚMERO_PERIODOS_DE_DURACIÓN_NUEVO', 'PERIODICIDAD_NUEVO', 'COSTO_MATRÍCULA_ESTUD_NUEVOS_NUEVO', 'PERIODICIDAD_ADMISIONES_NUEVO', 'FECHA_DE_REGISTRO_EN_SNIES_NUEVO', 'CINE_F_2013_AC_CAMPO_AMPLIO_NUEVO', 'CINE_F_2013_AC_CAMPO_ESPECÍFIC_NUEVO', 'CINE_F_2013_AC_CAMPO_DETALLADO_NUEVO', 'NÚCLEO_BÁSICO_DEL_CONOCIMIENTO_NUEVO', 'PROGINSTI_NUEVO', 'CÓDIGO_INSTITUCIÓN_ANTIGUO', 'NOMBRE_INSTITUCIÓN_ANTIGUO', 'SECTOR_ANTIGUO', 'DEPARTAMENTO_OFERTA_PROGRAMA_ANTIGUO', 'MUNICIPIO_OFERTA_PROGRAMA_ANTIGUO', 'NOMBRE_DEL_PROGRAMA_ANTIGUO', 'MODALIDAD_ANTIGUO', 'NÚMERO_CRÉDITOS_ANTIGUO', 'NÚMERO_PERIODOS_DE_DURACIÓN_ANTIGUO', 'PERIODICIDAD_ANTIGUO', 'COSTO_MATRÍCULA_ESTUD_NUEVOS_ANTIGUO', 'PERIODICIDAD_ADMISIONES_ANTIGUO', 'FECHA_DE_REGISTRO_EN_SNIES_ANTIGUO', 

# Agregación de Divisiones Uninorte

In [20]:
#Agregamos la división Uninorte a los programas a partir del CINE de Campo detallado en los 3 archivos
nuevosDFMerge    = nuevosDF.merge(on=["CINE_F_2013_AC_CAMPO_DETALLADO"], right=categories[["CINE_F_2013_AC_CAMPO_DETALLADO","DIVISIÓN UNINORTE"]], how="left")
eliminadosDFMerge = eliminadosDF.merge(on=["CINE_F_2013_AC_CAMPO_DETALLADO"], right=categories[["CINE_F_2013_AC_CAMPO_DETALLADO","DIVISIÓN UNINORTE"]], how="left")

# Para modificados: el merge de comparativa generó sufijos _NUEVO / _ANTIGUO en todas las columnas.
# Hacemos el join por la columna CINE con sufijo _NUEVO y luego normalizamos los nombres.
modificadosDFMerge = modificadosDF.merge(
    right=categories[["CINE_F_2013_AC_CAMPO_DETALLADO", "DIVISIÓN UNINORTE"]],
    left_on="CINE_F_2013_AC_CAMPO_DETALLADO_NUEVO",
    right_on="CINE_F_2013_AC_CAMPO_DETALLADO",
    how="left"
).drop(columns=["CINE_F_2013_AC_CAMPO_DETALLADO"])  # elimina la col duplicada que deja el right_on

# _NUEVO  → nombre original (valor actual del programa)
# _ANTIGUO → nombre_ANTERIOR (valor previo, para referencia)
rename_nuevo   = {c: c[:-6]              for c in modificadosDFMerge.columns if c.endswith("_NUEVO")}
rename_antiguo = {c: c[:-8] + "_ANTERIOR" for c in modificadosDFMerge.columns if c.endswith("_ANTIGUO")}
modificadosDFMerge = modificadosDFMerge.rename(columns={**rename_nuevo, **rename_antiguo})

modificadosDFMerge.head()

,CÓDIGO_INSTITUCIÓN,NOMBRE_INSTITUCIÓN,SECTOR,DEPARTAMENTO_OFERTA_PROGRAMA,MUNICIPIO_OFERTA_PROGRAMA,CÓDIGO_SNIES_DEL_PROGRAMA,NOMBRE_DEL_PROGRAMA,MODALIDAD,NÚMERO_CRÉDITOS,NÚMERO_PERIODOS_DE_DURACIÓN,...,COSTO_MATRÍCULA_ESTUD_NUEVOS_ANTERIOR,PERIODICIDAD_ADMISIONES_ANTERIOR,FECHA_DE_REGISTRO_EN_SNIES_ANTERIOR,CINE_F_2013_AC_CAMPO_AMPLIO_ANTERIOR,CINE_F_2013_AC_CAMPO_ESPECÍFIC_ANTERIOR,CINE_F_2013_AC_CAMPO_DETALLADO_ANTERIOR,NÚCLEO_BÁSICO_DEL_CONOCIMIENTO_ANTERIOR,PROGINSTI_ANTERIOR,FECHA_OBTENCION,DIVISIÓN UNINORTE
0,1711,UNIVERSIDAD DE LA SABANA,Privado,Cundinamarca,Chía,1239,DERECHO,Presencial,144,8.0,...,19271376.0,Semestral,1998-03-21 08:21:35,Administración de Empresas y Derecho,Derecho,Derecho,Derecho y afines,1239-194,07/04/2026,"Derecho, C. Política y Rel. Internacionales"
1,1711,UNIVERSIDAD DE LA SABANA,Privado,Cundinamarca,Chía,17489,INGENIERIA INFORMATICA,Presencial,162,9.0,...,18394277.0,Semestral,2021-08-05 00:00:00,Tecnologías de la Información y la Comunicació...,Tecnologías de la Información y la Comunicació...,Uso de computadores,"Ingeniería de sistemas, telemática y afines",17489-164,07/04/2026,Ingenierías
2,1811,UNIVERSIDAD LIBRE,Privado,Santander,Socorro,8410,CONTADURIA PUBLICA,Virtual,142,8.0,...,5072870.0,Semestral,1999-06-09 10:22:24,Administración de Empresas y Derecho,Educación comercial y administración,Contabilidad e impuestos,Contaduría pública,8410-142,07/04/2026,Escuela de Negocios
3,2737,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Privado,Risaralda,Pereira,5816,OPTOMETRIA,Presencial,173,10.0,...,8110000.0,Semestral,1998-03-21 08:23:13,Salud y Bienestar,Salud,Salud no clasificada en otra parte,"Optometría, otros programas de ciencias de la ...",5816-170,07/04/2026,Ciencias de la Salud


# Concatenación/Creación de Archivos Finales

In [21]:
# 1. Definición de rutas (Relativas para portabilidad)
output_folder = os.path.join(os.getcwd(), "Novedades_SNIES")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

path_inactivos = os.path.join(output_folder, "Inactivos pregrado.xlsx")
path_nuevos = os.path.join(output_folder, "Nuevos pregrado.xlsx")
path_modificados = os.path.join(output_folder, "Modificados pregrado.xlsx")

# Usaremos estas columnas para evitar duplicados:
# Esto evita que si corres el script 2 veces el mismo día se dupliquen los datos,
# pero permite guardar cambios de días diferentes.
columnas_deduplicacion = ['CÓDIGO_SNIES_DEL_PROGRAMA', 'FECHA_OBTENCION']

# --- PROCESO PARA INACTIVOS ---
if os.path.exists(path_inactivos):
    existing_df = pd.read_excel(path_inactivos)
    # Unificamos el nombre a eliminadosDFMerge para ser consistentes
    combined_df = pd.concat([existing_df, eliminadosDFMerge], ignore_index=True)
    final_df = combined_df.drop_duplicates(subset=columnas_deduplicacion)
    final_df.to_excel(path_inactivos, index=False)
else:
    eliminadosDFMerge.to_excel(path_inactivos, index=False)

# --- PROCESO PARA NUEVOS ---
if os.path.exists(path_nuevos):
    existing_df = pd.read_excel(path_nuevos)
    combined_df = pd.concat([existing_df, nuevosDFMerge], ignore_index=True)
    final_df = combined_df.drop_duplicates(subset=columnas_deduplicacion)
    final_df.to_excel(path_nuevos, index=False)
else:
    nuevosDFMerge.to_excel(path_nuevos, index=False)

# --- PROCESO PARA MODIFICADOS ---
if os.path.exists(path_modificados):
    existing_df = pd.read_excel(path_modificados)
    combined_df = pd.concat([existing_df, modificadosDFMerge], ignore_index=True)
    # En modificados, el código SNIES no tiene sufijo porque fue la llave del merge
    final_df = combined_df.drop_duplicates(subset=columnas_deduplicacion)
    final_df.to_excel(path_modificados, index=False)
else:
    modificadosDFMerge.to_excel(path_modificados, index=False)

print("✅ Archivos actualizados correctamente")

C:\Users\ARACATACA\AppData\Local\Temp\ipykernel_18792\2622411628.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([existing_df, eliminadosDFMerge], ignore_index=True)


✅ Archivos actualizados correctamente


# Agregaar estado de los programas 

In [22]:
# Leemos los archivos que acabamos de guardar usando las rutas dinámicas
data_inactivo = pd.read_excel(path_inactivos, sheet_name='Sheet1')
data_nuevos = pd.read_excel(path_nuevos, sheet_name='Sheet1')  
data_modificados = pd.read_excel(path_modificados, sheet_name='Sheet1')

# Aplicamos tu lógica de "Estado" (esta parte está perfecta)
for df_temp in [data_inactivo, data_nuevos, data_modificados]:
    df_temp["Estado"] = ["Activo" if x in droppedDf["CÓDIGO_SNIES_DEL_PROGRAMA"].values else "Inactivo" for x in df_temp["CÓDIGO_SNIES_DEL_PROGRAMA"]]

# Guardamos los cambios finales
data_inactivo.to_excel(path_inactivos, index=False)
data_nuevos.to_excel(path_nuevos, index=False)
data_modificados.to_excel(path_modificados, index=False)

print("✅ Todos los archivos se han actualizado en la carpeta Novedades_SNIES")

✅ Todos los archivos se han actualizado en la carpeta Novedades_SNIES


In [23]:
print(modificadosDF.columns.tolist())

['CÓDIGO_INSTITUCIÓN_NUEVO', 'NOMBRE_INSTITUCIÓN_NUEVO', 'SECTOR_NUEVO', 'DEPARTAMENTO_OFERTA_PROGRAMA_NUEVO', 'MUNICIPIO_OFERTA_PROGRAMA_NUEVO', 'CÓDIGO_SNIES_DEL_PROGRAMA', 'NOMBRE_DEL_PROGRAMA_NUEVO', 'MODALIDAD_NUEVO', 'NÚMERO_CRÉDITOS_NUEVO', 'NÚMERO_PERIODOS_DE_DURACIÓN_NUEVO', 'PERIODICIDAD_NUEVO', 'COSTO_MATRÍCULA_ESTUD_NUEVOS_NUEVO', 'PERIODICIDAD_ADMISIONES_NUEVO', 'FECHA_DE_REGISTRO_EN_SNIES_NUEVO', 'CINE_F_2013_AC_CAMPO_AMPLIO_NUEVO', 'CINE_F_2013_AC_CAMPO_ESPECÍFIC_NUEVO', 'CINE_F_2013_AC_CAMPO_DETALLADO_NUEVO', 'NÚCLEO_BÁSICO_DEL_CONOCIMIENTO_NUEVO', 'PROGINSTI_NUEVO', 'CÓDIGO_INSTITUCIÓN_ANTIGUO', 'NOMBRE_INSTITUCIÓN_ANTIGUO', 'SECTOR_ANTIGUO', 'DEPARTAMENTO_OFERTA_PROGRAMA_ANTIGUO', 'MUNICIPIO_OFERTA_PROGRAMA_ANTIGUO', 'NOMBRE_DEL_PROGRAMA_ANTIGUO', 'MODALIDAD_ANTIGUO', 'NÚMERO_CRÉDITOS_ANTIGUO', 'NÚMERO_PERIODOS_DE_DURACIÓN_ANTIGUO', 'PERIODICIDAD_ANTIGUO', 'COSTO_MATRÍCULA_ESTUD_NUEVOS_ANTIGUO', 'PERIODICIDAD_ADMISIONES_ANTIGUO', 'FECHA_DE_REGISTRO_EN_SNIES_ANTIGUO', 

In [24]:
# 1. Definimos qué columnas queremos vigilar
columnas_vigilar = [
    'MODALIDAD', 'NÚMERO_CRÉDITOS', 'COSTO_MATRÍCULA_ESTUD_NUEVOS', 
    'MUNICIPIO_OFERTA_PROGRAMA'
]

def identificar_cambios(row):
    cambios = []
    for col in columnas_vigilar:
        col_nueva = f"{col}_NUEVO"
        col_antigua = f"{col}_ANTIGUO"
        
        # Convertimos a string y quitamos espacios para comparar limpio
        val_n = str(row[col_nueva]).strip()
        val_a = str(row[col_antigua]).strip()
        
        if val_n != val_a:
            # Guardamos el nombre de la columna y el cambio (Ej: "MODALIDAD (Presencial -> Virtual)")
            cambios.append(f"{col.replace('_', ' ')}: {val_a} ➔ {val_n}")
    
    return " | ".join(cambios) if cambios else "Cambio en otros campos"

# Aplicamos la lógica a los modificados
if not modificadosDF.empty:
    modificadosDF['QUE_CAMBIO'] = modificadosDF.apply(identificar_cambios, axis=1)

In [23]:
data_modificados.head(15)

,CÓDIGO_INSTITUCIÓN,NOMBRE_INSTITUCIÓN,SECTOR,DEPARTAMENTO_OFERTA_PROGRAMA,MUNICIPIO_OFERTA_PROGRAMA,CÓDIGO_SNIES_DEL_PROGRAMA,NOMBRE_DEL_PROGRAMA,MODALIDAD,NÚMERO_CRÉDITOS,NÚMERO_PERIODOS_DE_DURACIÓN,...,NÚMERO_PERIODOS_DE_DURACIÓN_ANTIGUO,PERIODICIDAD_ANTIGUO,COSTO_MATRÍCULA_ESTUD_NUEVOS_ANTIGUO,PERIODICIDAD_ADMISIONES_ANTIGUO,FECHA_DE_REGISTRO_EN_SNIES_ANTIGUO,CINE_F_2013_AC_CAMPO_AMPLIO_ANTIGUO,CINE_F_2013_AC_CAMPO_ESPECÍFIC_ANTIGUO,CINE_F_2013_AC_CAMPO_DETALLADO_ANTIGUO,NÚCLEO_BÁSICO_DEL_CONOCIMIENTO_ANTIGUO,PROGINSTI_ANTIGUO
0,1102.0,UNIVERSIDAD NACIONAL DE COLOMBIA,Oficial,Antioquia,Medellín,127,ARQUITECTURA,Presencial,0.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
1,1102.0,UNIVERSIDAD NACIONAL DE COLOMBIA,Oficial,Antioquia,Medellín,124,INGENIERIA FORESTAL,Presencial,0.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
2,1104.0,UNIVERSIDAD NACIONAL DE COLOMBIA,Oficial,Valle del Cauca,Palmira,16904,INGENIERIA AGROINDUSTRIAL,Presencial,0.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
3,1104.0,UNIVERSIDAD NACIONAL DE COLOMBIA,Oficial,Valle del Cauca,Palmira,143,ZOOTECNIA,Presencial,0.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
4,1105.0,UNIVERSIDAD PEDAGOGICA NACIONAL,Oficial,Bogotá D.C.,"Bogotá, D.C.",10922,LICENCIATURA EN ARTES ESCENICAS,Presencial,160.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
5,1106.0,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...,Oficial,Boyacá,Tunja,10463,LICENCIATURA EN FILOSOFIA,Presencial,171.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
6,1106.0,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...,Oficial,Boyacá,Tunja,2683,MEDICINA,Presencial,287.0,13.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
7,1110.0,UNIVERSIDAD DEL CAUCA,Oficial,Cauca,Popayán,54801,MUSICA INSTRUMENTAL,Presencial,182.0,12.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
8,1111.0,UNIVERSIDAD TECNOLOGICA DE PEREIRA - UTP,Oficial,Risaralda,Pereira,13090,INGENIERIA ELECTRONICA,Presencial,184.0,10.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
9,1117.0,UNIVERSIDAD MILITAR-NUEVA GRANADA,Oficial,Bogotá D.C.,"Bogotá, D.C.",363,ADMINISTRACION DE EMPRESAS,Presencial,158.0,9.0,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN


In [25]:
from IPython.display import display, HTML

# Mejoramos la función con estilos inline para que se vea "bonita" en Outlook
cols_rename = {
    'NOMBRE_DEL_PROGRAMA_NUEVO': 'NOMBRE_DEL_PROGRAMA',
    'NOMBRE_INSTITUCIÓN_NUEVO': 'NOMBRE_INSTITUCIÓN',
    'MUNICIPIO_OFERTA_PROGRAMA_NUEVO': 'MUNICIPIO_OFERTA_PROGRAMA'
}
modificadosDFMerge_clean = modificadosDFMerge.rename(columns=cols_rename)
def preparar_tabla_html(df, color_header, color_letra_header="#FFFFFF", color_letra_celda="#FFFFFF"):
    if df.empty:
        return "<p style='color: #666;'>No se detectaron cambios en esta categoría.</p>"
    
    cols_interes = ['NOMBRE_DEL_PROGRAMA', 'NOMBRE_INSTITUCIÓN', 'MUNICIPIO_OFERTA_PROGRAMA']
    if 'QUE_CAMBIO' in df.columns:
        cols_interes.append('QUE_CAMBIO')

    # --- CONFIGURACIÓN DE COLORES Y ESTILOS ---
    estilos_tabla = (
        'border-collapse: collapse; '
        'width: 100%; '
        'font-family: Arial, sans-serif; '
        'font-size: 12px; '
        'margin-bottom: 20px;'
    )
    
    # Aquí cambiamos el color de la letra del encabezado (color_letra_header)
    estilos_th = (
        f'background-color: {color_header}; '
        f'color: {color_letra_header}; '
        'padding: 10px; '
        'text-align: left; '
        'border: 1px solid #ddd;'
    )
    
    # Aquí cambiamos el color de la letra de las celdas (color_letra_celda)
    estilos_td = (
        f'padding: 8px; '
        'border: 1px solid #ddd; '
        f'color: {color_letra_celda}; '
        'text-align: left;'
    )

    # Generación del HTML
    html = f'<table style="{estilos_tabla}">'
    html += '<thead><tr>'
    for col in cols_interes:
        html += f'<th style="{estilos_th}">{col.replace("_", " ")}</th>'
    html += '</tr></thead><tbody>'
    
    for _, row in df.head(10).iterrows():
        html += '<tr>'
        for col in cols_interes:
            html += f'<td style="{estilos_td}">{row[col]}</td>'
        html += '</tr>'
    
    html += '</tbody></table>'
    return html

# Generamos el cuerpo completo para la vista previa
today_str = datetime.now().strftime('%d/%m/%Y')
# Actualizamos la visualización del correo
cuerpo_email = f"""
<div style="font-family: Arial, sans-serif; color: #FFFFFF; max-width: 800px;">
    <h2 style="color: #003893; border-bottom: 3px solid #003893; padding-bottom: 10px;">
        Reporte de Inteligencia Académica SNIES
    </h2>
    <p>Hola, se ha procesado el análisis de competencia para hoy <strong>{today_str}</strong>:</p>
    
    <h3 style="color: #28a745;">🟢 NUEVOS PROGRAMAS (Nuevas amenazas)</h3>
    <p style="font-size: 12px;">Programas que aparecieron hoy en el SNIES.</p>
    {preparar_tabla_html(nuevosDF, "#28a745", "#FFFFFF", "#FFFFFF")}
    
    <h3 style="color: #dc3545;">🔴 PROGRAMAS INACTIVOS (Salieron del mercado)</h3>
    <p style="font-size: 12px;">Programas que ya no figuran como activos en la base de datos.</p>
    {preparar_tabla_html(eliminadosDF, "#dc3545", "#FFFFFF", "#FFFFFF")}
    
    <h3 style="color: #fd7e14;">🟠 MODIFICACIONES CRÍTICAS</h3>
    <p style="font-size: 12px;">Programas existentes que cambiaron sus condiciones (Precio, Créditos, etc.)</p>
    {preparar_tabla_html(modificadosDFMerge_clean, "#fd7e14", "#FFFFFF", "#FFFFFF")}
    
    <br>
    <p style="background-color: #f8f9fa; padding: 10px; border-left: 5px solid #003893; font-size: 11px;">
        <strong>Nota técnica:</strong> Los detalles completos están en los archivos Excel adjuntos. 
        En la tabla de modificaciones se resaltan los cambios detectados entre paréntesis (Antes ➔ Ahora).
    </p>
</div>
"""


# Visualizar en el Notebook
display(HTML(cuerpo_email))

NOMBRE DEL PROGRAMA,NOMBRE INSTITUCIÓN,MUNICIPIO OFERTA PROGRAMA
QUÍMICA,UNIVERSIDAD DE LA GUAJIRA,Riohacha
Contaduría Pública,UNIVERSIDAD CATOLICA DE MANIZALES,Manizales
NOMBRE DEL PROGRAMA,NOMBRE INSTITUCIÓN,MUNICIPIO OFERTA PROGRAMA
DERECHO,UNIVERSIDAD DE LA SABANA,Chía
INGENIERIA INFORMATICA,UNIVERSIDAD DE LA SABANA,Chía
CONTADURIA PUBLICA,UNIVERSIDAD LIBRE,Socorro
OPTOMETRIA,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Pereira


# Envío Automatizado de Correo

In [ ]:
#Cliente de Outlook
outlook = win32.Dispatch('outlook.application')

#Ruta de los archivos inactivos, nuevos y modificados
eliminados  =r"C:C:\Users\hasuarez\OneDrive - Universidad del Norte\General - EQUIPO IM PLANEACIÓN\PRACTICANTE\2025\2025-10\Novedades SNIES\Inactivos pregrado.xlsx"
nuevos =r"C:\Users\hasuarez\OneDrive - Universidad del Norte\General - EQUIPO IM PLANEACIÓN\PRACTICANTE\2025\2025-10\Novedades SNIES\Nuevos pregrado.xlsx"
modificados =r"C:\Users\hasuarez\OneDrive - Universidad del Norte\General - EQUIPO IM PLANEACIÓN\PRACTICANTE\2025\2025-10\Novedades SNIES\Modificados pregrado.xlsx"

#Correo único para todos los destinatarios
mail = outlook.CreateItem(0) #Creación del correo
mail.Subject = 'Nuevos/Inactivos/Modificados '+today #Asunto del correo
mail.HTMLBody = '<h2>Reporte mensual Programas Nuevos/Inactivos/Modificados pregrado</h2>' #Cuerpo del correo
mail.Attachments.Add(eliminados) #Agregamos los archivos al correo
mail.Attachments.Add(nuevos)
mail.Attachments.Add(modificados)
mail.To = 'mbastidas@uninorte.edu.co; degutierrez@uninorte.edu.co; lmlayes@uninorte.edu.co' #Todos los destinatarios
mail.Send() #Envío del correo                              

# Info Adicional: 

# Columnas originales


In [ ]:
todos = ['CÓDIGO_INSTITUCIÓN_PADRE',
 'CÓDIGO_INSTITUCIÓN',
 'NOMBRE_INSTITUCIÓN',
 'ESTADO_INSTITUCIÓN',
 'CARÁCTER_ACADÉMICO',
 'SECTOR',
 'REGISTRO_UNICO',
 'CÓDIGO_SNIES_DEL_PROGRAMA',
 'CÓDIGO_ANTERIOR_ICFES',
 'NOMBRE_DEL_PROGRAMA',
 'TITULO_OTORGADO',
 'ESTADO_PROGRAMA',
 'JUSTIFICACION',
 'JUSTIFICACION_DETALLADA',
 'RECONOCIMIENTO_DEL_MINISTERIO',
 'RESOLUCIÓN_DE_APROBACIÓN',
 'FECHA_DE_RESOLUCIÓN',
 'FECHA_EJECUTORIA',
 'VIGENCIA_AÑOS',
 'FECHA_DE_REGISTRO_EN_SNIES',
 'CINE_F_2013_AC_CAMPO_AMPLIO',
 'CINE_F_2013_AC_CAMPO_ESPECÍFIC',
 'CINE_F_2013_AC_CAMPO_DETALLADO',
 'ÁREA_DE_CONOCIMIENTO',
 'NÚCLEO_BÁSICO_DEL_CONOCIMIENTO',
 'NIVEL_ACADÉMICO',
 'NIVEL_DE_FORMACIÓN',
 'MODALIDAD',
 'NÚMERO_CRÉDITOS',
 'NÚMERO_PERIODOS_DE_DURACIÓN',
 'PERIODICIDAD',
 'SE_OFRECE_POR_CICLOS_PROPEDÉUT',
 'PERIODICIDAD_ADMISIONES',
 'PROGRAMA_EN_CONVENIO',
 'DEPARTAMENTO_OFERTA_PROGRAMA',
 'MUNICIPIO_OFERTA_PROGRAMA',
 'COSTO_MATRÍCULA_ESTUD_NUEVOS',
 'VIGENCIA TRANSITORIA',
 'OBSERVACIÓN DECRETO 1174/23']


# COlUMNAS ORIGINALES

print("Borradas:")
for column in todos:
    if column not in finalColumns:
        print(column)


Borradas:
CÓDIGO_INSTITUCIÓN_PADRE
ESTADO_INSTITUCIÓN
CARÁCTER_ACADÉMICO
REGISTRO_UNICO
CÓDIGO_ANTERIOR_ICFES
TITULO_OTORGADO
ESTADO_PROGRAMA
JUSTIFICACION
JUSTIFICACION_DETALLADA
RECONOCIMIENTO_DEL_MINISTERIO
RESOLUCIÓN_DE_APROBACIÓN
FECHA_DE_RESOLUCIÓN
FECHA_EJECUTORIA
VIGENCIA_AÑOS
ÁREA_DE_CONOCIMIENTO
NIVEL_ACADÉMICO
NIVEL_DE_FORMACIÓN
SE_OFRECE_POR_CICLOS_PROPEDÉUT
PROGRAMA_EN_CONVENIO
VIGENCIA TRANSITORIA
OBSERVACIÓN DECRETO 1174/23
